# Fase 2: Filtros y Reducción de Dimensionalidad
En este cuaderno tomaremos la base de datos explorada y construiremos cuatro escenarios distintos de características (features) para evaluar posteriormente cuál ofrece el mejor rendimiento predictivo.

1. **Escenario 1 (Crudo):** Todas las características originales estandarizadas.
2. **Escenario 2 (Filtro Manual):** Eliminación de variables con alta colinealidad (>0.85) o bajo poder predictivo (<0.05).
3. **Escenario 3 (Selección LASSO):** Selección algorítmica incrustada usando penalización L1.
4. **Escenario 4 (PCA):** Reducción de dimensionalidad extrayendo los componentes principales que expliquen el 95% de la varianza.

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos y Estandarización Base (Escenario 1)
Para esta fase de transformación, partimos de la **base de datos original e intacta** (cargada desde nuestro archivo local `dataset_cancer.csv` para agilizar el proceso). 

Dado que los algoritmos que utilizaremos para la selección y reducción (LASSO y PCA) son altamente sensibles a la magnitud geométrica de los datos, es **estrictamente necesario estandarizar** todas las variables (Z-score) antes de aplicar cualquier método. Esta matriz estandarizada completa será nuestro Escenario 1.

In [19]:
# Cargar el dataset desde la carpeta local
df = pd.read_csv('./data/dataset_cancer.csv')

# Separar variables independientes (X) y el objetivo (y)
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

# Estandarización (Z-score)
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

# Convertir de vuelta a DataFrame para no perder los nombres de las columnas
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

# Renombrar conceptualmente para claridad del proyecto
X_escenario_1 = X_scaled 

print("Datos cargados y estandarizados.")
print(f"Shape del Escenario 1 (Datos Crudos): {X_escenario_1.shape}")

Datos cargados y estandarizados.
Shape del Escenario 1 (Datos Crudos): (569, 30)


## 2. Escenario 2: Filtro Manual (Heurístico)
Basados en el Análisis Exploratorio (EDA) del cuaderno anterior, implementamos un algoritmo que evalúa la multicolinealidad y el poder predictivo simultáneamente:
1. `umbral_colinealidad = 0.85`: Si dos variables están altamente correlacionadas, se elimina aquella que tenga menor correlación con el diagnóstico (la variable objetivo).
2. `umbral_ruido = 0.05`: Si una variable tiene una correlación casi nula con el diagnóstico, se descarta por ser considerada ruido estadístico.

In [20]:
# 1. Definir umbrales matemáticos
umbral_colinealidad = 0.85  # Si dos variables se parecen más de esto, una debe irse
umbral_ruido = 0.05         # Si una variable tiene menos de esto de poder predictivo, es basura

# 2. Obtener el poder predictivo (Correlación absoluta con Diagnosis)
# Unimos temporalmente X_scaled e y para calcular su correlación
df_temp = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)
corr_objetivo = df_temp.corr()['Diagnosis'].drop('Diagnosis').abs()

# 3. Calcular la matriz de correlación solo entre las predictoras (absoluta)
matriz_corr_X = X_scaled.corr().abs()

# Lista para guardar las variables condenadas a ser eliminadas
columnas_a_eliminar = set()

# 4. ALGORITMO DE ELIMINACIÓN POR MULTICOLINEALIDAD
# Recorrer el triángulo superior de la matriz para evaluar cada par una sola vez
for i in range(len(matriz_corr_X.columns)):
    for j in range(i+1, len(matriz_corr_X.columns)):
        var1 = matriz_corr_X.columns[i]
        var2 = matriz_corr_X.columns[j]
        
        # Si la correlación entre ellas supera el umbral y ninguna ha sido eliminada aún
        if matriz_corr_X.iloc[i, j] > umbral_colinealidad:
            # Eliminar la que tenga MENOR poder predictivo con el diagnóstico
            if corr_objetivo[var1] < corr_objetivo[var2]:
                columnas_a_eliminar.add(var1)
            else:
                columnas_a_eliminar.add(var2)

# 5. ALGORITMO DE ELIMINACIÓN POR RUIDO ESTADÍSTICO
for col in X_scaled.columns:
    if corr_objetivo[col] < umbral_ruido:
        columnas_a_eliminar.add(col)

columnas_a_eliminar = list(columnas_a_eliminar)

# 6. Ejecutar la limpieza para crear el Escenario 2
X_manual = X_scaled.drop(columns=columnas_a_eliminar)

print(f"Dimensiones de Escenario 1 (Crudo): {X_scaled.shape}")
print(f"Dimensiones de Escenario 2 (Filtro Manual): {X_manual.shape}")
print(f"Cantidad de variables eliminadas: {len(columnas_a_eliminar)}\n")

print("=== VARIABLES ELIMINADAS ===")
for col in sorted(columnas_a_eliminar):
    print(f"- {col}")

Dimensiones de Escenario 1 (Crudo): (569, 30)
Dimensiones de Escenario 2 (Filtro Manual): (569, 14)
Cantidad de variables eliminadas: 16

=== VARIABLES ELIMINADAS ===
- area1
- area2
- area3
- compactness1
- compactness3
- concave_points1
- concavity1
- concavity3
- fractal_dimension1
- perimeter1
- perimeter2
- radius1
- radius3
- symmetry2
- texture1
- texture2


## 3. Escenario 3: Selección Incrustada con LASSO
En lugar de filtrar matemáticamente variable por variable, delegaremos la tarea a la regularización **L1 (LASSO)**. Entrenaremos una Regresión Logística con un nivel de penalización estricto (`C=1`) y usaremos `SelectFromModel` para extraer únicamente las variables cuyo coeficiente matemático haya sobrevivido (sea distinto de 0).

In [21]:
# Configurar la Regresión Logística con penalización LASSO (L1)
modelo_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1, random_state=30)

# Selector de características basado en el modelo
selector = SelectFromModel(modelo_lasso)
selector.fit(X_scaled, y)

# Obtener los nombres de las variables que sobrevivieron
variables_lasso = X_scaled.columns[selector.get_support()]

# Crear el Escenario 3
X_lasso = X_scaled[variables_lasso]

print(f"Variables descartadas por LASSO: {X_scaled.shape[1] - len(variables_lasso)}")
print(f"Variables sobrevivientes ({len(variables_lasso)}):")
for var in variables_lasso:
    print(f"- {var}")
    
print(f"\nShape del Escenario 3 (LASSO): {X_lasso.shape}")

Variables descartadas por LASSO: 15
Variables sobrevivientes (15):
- concavity1
- concave_points1
- fractal_dimension1
- radius2
- texture2
- smoothness2
- compactness2
- fractal_dimension2
- radius3
- texture3
- area3
- smoothness3
- concavity3
- concave_points3
- symmetry3

Shape del Escenario 3 (LASSO): (569, 15)


## 4. Escenario 4: Reducción de Dimensionalidad (PCA)
A diferencia de los métodos anteriores, PCA no elimina variables, sino que las fusiona en un nuevo espacio geométrico. Extraeremos los Componentes Principales necesarios para encapsular el **95% de la información (varianza)** original de los tumores, eliminando la colinealidad al 100% pero perdiendo la interpretabilidad directa de las características.

In [22]:
# Inicializar PCA para conservar el 95% de la varianza
pca = PCA(n_components=0.95, random_state=30)

# Ajustar y transformar los datos
X_pca_array = pca.fit_transform(X_scaled)

# Crear nombres para los nuevos componentes (PC1, PC2, etc.)
nombres_pca = [f"PC{i+1}" for i in range(X_pca_array.shape[1])]
X_pca = pd.DataFrame(X_pca_array, columns=nombres_pca)

print(f"Componentes necesarios para explicar el 95% de la varianza: {pca.n_components_}")
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_) * 100:.2f}%\n")
print(f"Shape del Escenario 4 (PCA): {X_pca.shape}")

Componentes necesarios para explicar el 95% de la varianza: 10
Varianza total explicada: 95.16%

Shape del Escenario 4 (PCA): (569, 10)


## 5. Exportación de Escenarios
Finalmente, guardamos los 4 escenarios de características (junto con la variable objetivo `y`) en la carpeta `data/` para ser consumidos directamente por los algoritmos predictivos en la fase de **Estudio Comparativo (Notebook 03)**.

In [23]:
# Guardar los DataFrames transformados a CSV (index=False para no guardar la columna de números de fila)
X_escenario_1.to_csv('./data/X_escenario_1_crudo.csv', index=False)
X_manual.to_csv('./data/X_escenario_2_manual.csv', index=False)
X_lasso.to_csv('./data/X_escenario_3_lasso.csv', index=False)
X_pca.to_csv('./data/X_escenario_4_pca.csv', index=False)
y.to_csv('./data/y_target.csv', index=False)

print("¡Exportación de los 4 escenarios completada con éxito!")

¡Exportación de los 4 escenarios completada con éxito!
